# Week 16: Hypothesis Testing

## 1 -  Types of Inference
There are three types of inference we can make with regards to a parameter of interest:
- *Point Estimate* - A function of the data aiming to be as close as possible to true value paramater (e.g. estimating the true mean wind speed as the mean of observed wind speeds). This was covered in Week 14.
- *Interval Estimate* - A region containing the true parameter value with some set probability, usually defined by a point estimate and its variance.
- *Hypothesis Testing* - A test of whether the data has come from a population whose distribution is consistent with some null hypothesis

There are two main inference methods: classical (frequentist) and Bayesian. The frequentist approach assumes all unknown parameters are fixed, whereas Bayesian approaches treat them as random variables which take on certain values with set probabilities. We can think as frequentist as just 'all about the data'. We don't make many assumptions and make inferences based on what we observe. Conversely, Bayesian approaches make *prior* assumptions about the distribution, probability or other properties of the parameters. The observations we make are then used to *update* these assumptions to form what we call a *posterior*. Which type of approach we takes affects both the method and the interpretation of inferences.


## 2.1 - Hypothesis Testing

Hypothesis testing allows us to determine statistically whether an assumption or theory, known as the *null hypothesis*, is sufficiently evidenced by data. It provides us with specific rules and thresholds to determine whether to accept or reject our assumptio, $H_0$. A hypothesis can be *simple*, if it fully describes the theorised distribution of the data, or *composite* otherwise. For example, the assumption that the data is Normally distributed is a composite hypothesis, whilst the hypothesis that a dataset is uniform on [0, $\pi$] is simple.

To carry out the hypothesis test we first propose a null hypothesis, $H_0$, about the distribution of our data. Importantly, we must specify this without using the data to inform our assumptions. Secondly, we specify an alternative hypothesis, $H_1$. The simplest alternative hypothesis is 'not $H_1$'. For example, if our hypothesis is that the data has mean 5, the alternative hypothesis is that the data does not have mean 5. Other alternative hypotheses may specify how the data deviates from $H_0$. For example, if we wish to test for improvement between two data sets, the null hypothesis may be $\mu_2=\mu_1$, with alternative hypothesis $\mu_2>\mu_1$. In a hypothesis test we have two possible outcomes:
- Reject $H_0$ in favour of $H_1$
- Do not reject $H_0$ in favour of $H_1$

To determine the outcome a test will specify a **critical region**, $C$, such that if the data, $x$, falls within the critical region,
$$x\in C,$$
then we reject $H_0$ in favour of $H_1$, otherwise we accept $H_0$.hIt is common to define the critical region in terms of a **function of the data**, i.e. $$C=\{x: T(x)\in\mathcal{T}\},$$
where the statistic $T$ is a function of the data known as the **test statistic**. There are two types of error when we carry out a hypothesis test:
- **Type I error**: Reject $H_0$ when $H_0$ is true
- **Type II error**: Do not reject $H_0$ when $H_0$ is false

These two errors determine two important characteristics of the test:
- The size, $\alpha$ is the probability of a type I error
- The power, $\beta$, is 1 - the probability of a type II error
Typically we pre-specify the probability of type I error, $\alpha$, which determines the critical region, whilst we try to minimise the probability of type two error (i.e. try to maximise the power of the test).

The **p-value** allows us to quantify the strength of evidence against the null hypothesis (or alternatively the strength of evidence for rejecting the null hypothesis). The smaller the p-value the stronger the evidence is to reject $H_0$. We can consider it to be the probability that we observe something as extreme as our data under the null hypothsis, therefore the less likely our observations are under the assumption of the null hypothesis, the less likely it is that the hypothesis describes the data correctly.

### 2.2 - Types of Hypothesis test
The two most common applications for hypothesis testing in wind energy are to test goodness-of-fit, for example distribution fitting with reliability or climate data, or to test differences in data means, for example testing whether power yields are improved or decreased after a certain upgrade, whether wind speed is higher at one site than another, or whether a measurement instrument has degraded. These require different types of hypothesis tests.

- **Goodness-of-fit tests**: to determine how well a certain distribution describes the data. Examples include the chi-square test and the Kolmogorov-Smirnov test.
- **Mean-comparison tests**: to determine whether the mean of two distributions are the same. Importantly, one-sided tests allow us to test for improvement or degradation. Examples include classic (student's) t-test, Welch's t-test and z-test. An ANOVA test allows us to extend to three or more samples.

Each of these tests has its own assumptions, and we must determine whether these assumptions are valid for the data we are testing in order to select the most appropriate test.


|Test|Purpose|Assumptions|
|--|--|--|
|student's t-test|Mean comparison|Continuous data, independent observations, approximately normally distributed data, equal variance between samples|
|Welch's t-test|Mean comparison|Continuous data, independent observations, approximately normally distributed data|
|z-test|Mean comparison|Continuous data, known population variance, independent observations and samples, approximately normally distributed data or sufficiently large sample size for CLT|
Chi-square test|GOF testing|Data are counts/frequencies (categorical), mutually exclusive categories, min 5 frequency per category, random sampling, independence|
Kolmogorov-Smirnov test|GOF testing|Continuous, independent|

### 2.2.1 - Goodness of Fit test example
Let us look at an example of how we may carry out goodness-of-fit tests in Python. First, we import our example data as usual, and fit a weibull distribution to the wind speed as learnt in our distribution fitting tutorial.

In [22]:
import pandas as pd
import numpy as np
from scipy.stats import weibull_min as weibull

data = pd.read_csv('example_data.csv')
wind_speed = data['Wind Speed (m/s)']

fitted_dist = weibull.fit(wind_speed)
k, loc, scale = fitted_dist

What is returned is the three parameters of the Weibull distribution. To check if it matches the fitted distribution, we can generate a cdf of values from the true Weibull distribution and compare the two samples.

In [45]:
# Generate random variables with the 
generated_cdf = weibull.rvs(k, loc, scale, 1000)

We may now carry out a Kolmogorov-Smirnov test between the two samples to test whether the samples come from the same distribution. The null hypothesis in this case is that the two samples are drawn from the same distribution, whilst the alternative hypothesis is that the two samples are not drawn from the same distribution. This means the test is **two-sided**.

In [71]:
from scipy.stats import kstest
alpha = 0.05
statistic, pval = kstest(wind_speed, generated_cdf)
print(f'P-value = {np.round(pval,3)}')
if pval >= alpha:
    print(f'Insufficient evidence to reject H0 at confidence level {(1-alpha)*100}%')
else:
    print(f'Sufficient evidence to reject H0 in favour of H1 at confidence level {(1-alpha)*100}%')

P-value = 0.286
Insufficient evidence to reject H0 at confidence level 95.0%


Remember that the interpretation of the p-value is the likelihood of observing values as extreme as those seen in the data. Therefore, the smaller the p-value the more evidence we have to reject $H_0$. If the p-value is **below** the size ($\alpha$) then we have sufficient evidence to reject $H_0$ in favour of $H_1$, and otherwise we do not have sufficient evidence to reject $H_0$.

#### Exercise 2.2.1.1
Redo this test at confidence level 99% - do we have the same result?
What about at 75%?

### 2.2.2 - Mean comparison example
Let us now test whether wind speed is higher in the winter months than the summer months. First we split the data into two categories.

In [63]:
data['datetime'] = pd.to_datetime(data['Date/Time'], format='mixed')
data['month'] = [d.month for d in data.datetime]
season_dict = {1:'winter', 2:'winter', 3:'spring', 4:'spring', 5:'spring', 6:'summer', 7:'summer', 8:'summer', 9:'autumn', 10:'autumn', 11:'autumn', 12:'winter'}
data['season'] = [season_dict[m] for m in data.month]

In [64]:
winter_speeds = data[data.season=='winter']['Wind Speed (m/s)']
summer_speeds = data[data.season=='summer']['Wind Speed (m/s)']

We may now test whether winter wind speeds are higher than summer wind speeds by testing the null hypothesis, that the means of the two samples are the same, against a **one-sided** alternative hypothesis, that the wind speeds in winter are greater than the wind speeds in summer.

In [70]:
from scipy.stats import ttest_ind as ttest

alpha = 0.05
statistic, pval = ttest(winter_speeds, summer_speeds, alternative='greater')

print(f'P-value = {np.round(pval,3)}')
if pval >= alpha:
    print(f'Insufficient evidence to reject H0 at confidence level {(1-alpha)*100}%')
else:
    print(f'Sufficient evidence to reject H0 in favour of H1 at confidence level {(1-alpha)*100}%')

P-value = 0.0
Sufficient evidence to reject H0 in favour of H1 at confidence level 95.0%


We see we have sufficient evidence to reject our null hypothesis in favour of the hypothesis that wind speeds are greater in winter than in summer. This test is very powerful as we can use it to test for whether we have improvement amongst two scenarios with a set confidence in a statistically rigorous way. We can use this to test the impact of certain interventions such as upgrades on our sites.

## 3 - Confidence and Credible Intervals
A confidence interval (frequentist) gives a range of values that would contain the true population parameter a certain percentage of the time if the experiment were repeated many times. A 95% CI does not mean there is a 95% probability the true parameter is inside the interval, but that if we repeated the sampling many times, 95% of the constructed intervals would contain the true parameter.
A Credible Intervals (Bayesian) gives a range of parameter values containing a certain proportion of the posterior distribution. A 95% credible interval means that there is a 95% probability that the true parameter lies within the interval given the data and the prior.

|        | Confidence Interval (Frequentist)                            | Credible Interval (Bayesian)                   |
| --- | ------------------------------------------------------------ | ---------------------------------------------- |
| Interpretation | Long-run frequency: intervals that *would* contain the truth | Direct probability: truth *is* in the interval |
| Based on       | Sampling distribution                                        | Posterior distribution                         |
| Requires       | No prior                                                     | Prior + likelihood                             |
| Computed via   | t/z distribution formulas                                    | Posterior percentiles (MCMC or analytic)   |
The simplest way to compute a confidence interval is using a point estimate and a standard error. For a symmetrical distribution such as the normal distribution we can construct a confidence interval is as $[\theta-C\cdot SE, \theta+C\cdot SE]$, where
$$SE = \frac{\sigma}{\sqrt{n}},$$
with $\sigma$ representing the standard deviation, $n$ being the observed sample size, and $C$ the *critical value*. The critical value is defined by the confidence level (e.g. 95%, 99%), and the distribution. The most common one is the normal-distribution at level 95% which is 1.96. Other values can be found online, using statistical tables or using Python code from in-built packages. For example, for the normal distribution we can use the package scipy.stats.norm and use the ppf to find the related percentile at the size of interest, $\alpha$.

In [10]:
from scipy.stats import norm

# set confidence level
a = 0.95

# Normal distribution
z = norm.ppf(a)

Let's use this value $z$ that we have calculated to generate a confidence interval for the mean wind speed from our example data set. First we import our data as usual.

We first form our point estimate for the mean as the sample mean. We may remember that this type of estimator is consistent and unbiased, so it is a good choice for our central value.

In [6]:
import numpy as np
mean_wind_speed = np.mean(wind_speed)

7.557952236083194

We then calculate the standard error as the standard deviation divided by the root of the sample size.

In [8]:
SE = np.std(wind_speed)/len(wind_speed)

We now have all the tools we need to build a 95% confidence interval.

In [13]:
CI_lower = mean_wind_speed - SE*z
CI_upper = mean_wind_speed + SE*z

print(f'CI = [{np.round(CI_lower,4)}, {np.round(CI_upper, 4)}]')

CI = [7.5578, 7.5581]


We can see in this case our interval is very tight around the mean value. The things that affect the width of our confidence interval are:
- Critical value ($z$)
- Standard deviation ($\sigma$)
- Sample size ($n$)

Try changing each of these and seeing how the confidence interval changes.

#### Exercise 3.1
Calculate the confidence interval at size 75%

#### Exercise 3.2
Rescale the standard deviation to be 10 using what you learnt in the feature engineering tutorial, and recalculate the confidence interval at 95%.

#### Exercise 3.3
Take a subset of the first 1000 data points, and recalculate the confidence interval at 95%. 

#### Exercise 3.4 
Why might taking a subset of the data points not just reflect what changes with sample size? What else might be changing and how can we adjust for that?